<a href="https://colab.research.google.com/github/ancestor9/Introduction-to-BigData-Analysis/blob/main/12%EC%A3%BC%EC%B0%A8/LLM_working_with_strings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectorized String Operations

## **[PythonDataScienceHandbook 코드](https://jakevdp.github.io/PythonDataScienceHandbook/03.10-working-with-strings.html)**

One strength of Python is its relative ease in handling and manipulating string data.
Pandas builds on this and provides a comprehensive set of *vectorized string operations* that are an important part of the type of munging required when working with (read: cleaning up) real-world data.
In this chapter, we'll walk through some of the Pandas string operations, and then take a look at using them to partially clean up a very messy dataset of recipes collected from the internet.

## Introducing Pandas String Operations

We saw in previous chapters how tools like NumPy and Pandas generalize arithmetic operations so that we can easily and quickly perform the same operation on many array elements. For example:

In [1]:
import numpy as np
x = np.array([2, 3, 5, 7, 11, 13])
x * 2

array([ 4,  6, 10, 14, 22, 26])

This *vectorization* of operations simplifies the syntax of operating on arrays of data: we no longer have to worry about the size or shape of the array, but just about what operation we want done.
For arrays of strings, NumPy does not provide such simple access, and thus you're stuck using a more verbose loop syntax:

In [2]:
data = ['peter', 'Paul', 'MARY', 'gUIDO']
[s.capitalize() for s in data]

['Peter', 'Paul', 'Mary', 'Guido']

This is perhaps sufficient to work with some data, but it will break if there are any missing values, so this approach requires putting in extra checks:

In [3]:
data = ['peter', 'Paul', None, 'MARY', 'gUIDO']
[s if s is None else s.capitalize() for s in data]

['Peter', 'Paul', None, 'Mary', 'Guido']

This kind of manual approach is not only verbose and inconvenient, it can be error-prone.

Pandas includes features to address both this need for vectorized string operations and the need for correctly handling missing data via the `str` attribute of Pandas `Series` and `Index` objects containing strings.
So, for example, if we create a Pandas `Series` with this data we can directly call the `str.capitalize` method, which has missing value handling built in:

In [4]:
import pandas as pd
names = pd.Series(data)
names.str.capitalize()

,0
0,Peter
1,Paul
2,None
3,Mary
4,Guido


## Tables of Pandas String Methods

If you have a good understanding of string manipulation in Python, most of the Pandas string syntax is intuitive enough that it's probably sufficient to just list the available methods. We'll start with that here, before diving deeper into a few of the subtleties.
The examples in this section use the following `Series` object:

In [5]:
monte = pd.Series(['Graham Chapman', 'John Cleese', 'Terry Gilliam',
                   'Eric Idle', 'Terry Jones', 'Michael Palin'])

### Methods Similar to Python String Methods

Nearly all of Python's built-in string methods are mirrored by a Pandas vectorized string method. Here is a list of Pandas `str` methods that mirror Python string methods:

|           |                |                |                |
|-----------|----------------|----------------|----------------|
|`len()`    | `lower()`      | `translate()`  | `islower()`    |
|`ljust()`  | `upper()`      | `startswith()` | `isupper()`    |
|`rjust()`  | `find()`       | `endswith()`   | `isnumeric()`  |
|`center()` | `rfind()`      | `isalnum()`    | `isdecimal()`  |
|`zfill()`  | `index()`      | `isalpha()`    | `split()`      |
|`strip()`  | `rindex()`     | `isdigit()`    | `rsplit()`     |
|`rstrip()` | `capitalize()` | `isspace()`    | `partition()`  |
|`lstrip()` | `swapcase()`   | `istitle()`    | `rpartition()` |

Notice that these have various return values. Some, like `lower`, return a series of strings:

In [6]:
monte.str.lower()

,0
0,graham chapman
1,john cleese
2,terry gilliam
3,eric idle
4,terry jones
5,michael palin


But some others return numbers:

In [7]:
monte.str.len()

,0
0,14
1,11
2,13
3,9
4,11
5,13


Or Boolean values:

In [8]:
monte.str.startswith('T')

,0
0,False
1,False
2,True
3,False
4,True
5,False


Still others return lists or other compound values for each element:

In [9]:
monte.str.split()

,0
0,"[Graham, Chapman]"
1,"[John, Cleese]"
2,"[Terry, Gilliam]"
3,"[Eric, Idle]"
4,"[Terry, Jones]"
5,"[Michael, Palin]"


We'll see further manipulations of this kind of series-of-lists object as we continue our discussion.

### Methods Using Regular Expressions

In addition, there are several methods that accept regular expressions (regexps) to examine the content of each string element, and follow some of the API conventions of Python's built-in `re` module:

| Method    | Description |
|-----------|-------------|
| `match`   | Calls `re.match` on each element, returning a Boolean. |
| `extract` | Calls `re.match` on each element, returning matched groups as strings.|
| `findall` | Calls `re.findall` on each element |
| `replace` | Replaces occurrences of pattern with some other string|
| `contains`| Calls `re.search` on each element, returning a boolean |
| `count`   | Counts occurrences of pattern|
| `split`   | Equivalent to `str.split`, but accepts regexps |
| `rsplit`  | Equivalent to `str.rsplit`, but accepts regexps |

With these, we can do a wide range of operations.
For example, we can extract the first name from each element by asking for a contiguous group of characters at the beginning of each element:

In [10]:
monte.str.extract('([A-Za-z]+)', expand=False)

,0
0,Graham
1,John
2,Terry
3,Eric
4,Terry
5,Michael


Or we can do something more complicated, like finding all names that start and end with a consonant, making use of the start-of-string (`^`) and end-of-string (`$`) regular expression characters:

In [11]:
monte.str.findall(r'^[^AEIOU].*[^aeiou]$')

,0
0,[Graham Chapman]
1,[]
2,[Terry Gilliam]
3,[]
4,[Terry Jones]
5,[Michael Palin]


The ability to concisely apply regular expressions across `Series` or `DataFrame` entries opens up many possibilities for analysis and cleaning of data.

### Miscellaneous Methods
Finally, there are some miscellaneous methods that enable other convenient operations:

| Method | Description |
|--------|-------------|
| `get` | Indexes each element |
| `slice` | Slices each element|
| `slice_replace` | Replaces slice in each element with the passed value|
| `cat`      | Concatenates strings|
| `repeat` | Repeats values |
| `normalize` | Returns Unicode form of strings |
| `pad` | Adds whitespace to left, right, or both sides of strings|
| `wrap` | Splits long strings into lines with length less than a given width|
| `join` | Joins strings in each element of the `Series` with the passed separator|
| `get_dummies` | Extracts dummy variables as a `DataFrame` |

#### Vectorized item access and slicing

The `get` and `slice` operations, in particular, enable vectorized element access from each array.
For example, we can get a slice of the first three characters of each array using `str.slice(0, 3)`.
Note that this behavior is also available through Python's normal indexing syntax; for example, `df.str.slice(0, 3)` is equivalent to `df.str[0:3]`:

In [12]:
monte.str[0:3]

,0
0,Gra
1,Joh
2,Ter
3,Eri
4,Ter
5,Mic


Indexing via `df.str.get(i)` and `df.str[i]` are likewise similar.

These indexing methods also let you access elements of arrays returned by `split`.
For example, to extract the last name of each entry, we can combine `split` with `str` indexing:

In [13]:
monte.str.split().str[-1]

,0
0,Chapman
1,Cleese
2,Gilliam
3,Idle
4,Jones
5,Palin


#### Indicator variables

Another method that requires a bit of extra explanation is the `get_dummies` method.
This is useful when your data has a column containing some sort of coded indicator.
For example, we might have a dataset that contains information in the form of codes, such as A = "born in America," B = "born in the United Kingdom," C = "likes cheese," D = "likes spam":

In [14]:
full_monte = pd.DataFrame({'name': monte,
                           'info': ['B|C|D', 'B|D', 'A|C',
                                    'B|D', 'B|C', 'B|C|D']})
full_monte

,name,info
0,Graham Chapman,B|C|D
1,John Cleese,B|D
2,Terry Gilliam,A|C
3,Eric Idle,B|D
4,Terry Jones,B|C
5,Michael Palin,B|C|D


The `get_dummies` routine lets us split out these indicator variables into a `DataFrame`:

In [15]:
full_monte['info'].str.get_dummies('|')

,A,B,C,D
0,0,1,1,1
1,0,1,0,1
2,1,0,1,0
3,0,1,0,1
4,0,1,1,0
5,0,1,1,1


With these operations as building blocks, you can construct an endless range of string processing procedures when cleaning your data.

We won't dive further into these methods here, but I encourage you to read through ["Working with Text Data"](https://pandas.pydata.org/pandas-docs/stable/user_guide/text.html) in the Pandas online documentation, or to refer to the resources listed in [Further Resources](03.13-Further-Resources.ipynb).

## Example: Recipe Database

These vectorized string operations become most useful in the process of cleaning up messy, real-world data.
Here I'll walk through an example of that, using an open recipe database compiled from various sources on the web.
Our goal will be to parse the recipe data into ingredient lists, so we can quickly find a recipe based on some ingredients we have on hand. The scripts used to compile this can be found at https://github.com/fictivekin/openrecipes, and the link to the most recent version of the database is found there as well.

This database is about 30 MB, and can be downloaded and unzipped with these commands:

In [16]:
!mkdir -p data  # 구글클라우드 현재 폴더에서 data 폴더 만들기

In [17]:
repo = "https://raw.githubusercontent.com/jakevdp/open-recipe-data/master"
!cd data && curl -O {repo}/recipeitems.json.gz
!gunzip data/recipeitems.json.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 29.3M  100 29.3M    0     0  32.2M      0 --:--:-- --:--:-- --:--:-- 32.2M
gzip: data/recipeitems.json already exists; do you wish to overwrite (y or n)? y


The database is in JSON format, so we will use `pd.read_json` to read it (`lines=True` is required for this dataset because each line of the file is a JSON entry):

In [18]:
recipes = pd.read_json('data/recipeitems.json', lines=True)
recipes.shape

(173278, 17)

In [19]:
recipes.head()

,_id,name,ingredients,url,image,ts,cookTime,source,recipeYield,datePublished,prepTime,description,totalTime,creator,recipeCategory,dateModified,recipeInstructions
0,{'$oid': '5160756b96cc62079cc2db15'},Drop Biscuits and Sausage Gravy,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,http://thepioneerwoman.com/cooking/2013/03/dro...,http://static.thepioneerwoman.com/cooking/file...,{'$date': 1365276011104},PT30M,thepioneerwoman,12,2013-03-11,PT10M,"Late Saturday afternoon, after Marlboro Man ha...",NaN,NaN,NaN,NaN,NaN
1,{'$oid': '5160756d96cc62079cc2db16'},Hot Roast Beef Sandwiches,12 whole Dinner Rolls Or Small Sandwich Buns (...,http://thepioneerwoman.com/cooking/2013/03/hot...,http://static.thepioneerwoman.com/cooking/file...,{'$date': 1365276013902},PT20M,thepioneerwoman,12,2013-03-13,PT20M,"When I was growing up, I participated in my Ep...",NaN,NaN,NaN,NaN,NaN
2,{'$oid': '5160756f96cc6207a37ff777'},Morrocan Carrot and Chickpea Salad,Dressing:\n1 tablespoon cumin seeds\n1/3 cup /...,http://www.101cookbooks.com/archives/moroccan-...,http://www.101cookbooks.com/mt-static/images/f...,{'$date': 1365276015332},NaN,101cookbooks,NaN,2013-01-07,PT15M,A beauty of a carrot salad - tricked out with ...,NaN,NaN,NaN,NaN,NaN
3,{'$oid': '5160757096cc62079cc2db17'},Mixed Berry Shortcake,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,http://thepioneerwoman.com/cooking/2013/03/mix...,http://static.thepioneerwoman.com/cooking/file...,{'$date': 1365276016700},PT15M,thepioneerwoman,8,2013-03-18,PT15M,It's Monday! It's a brand new week! The birds ...,NaN,NaN,NaN,NaN,NaN
4,{'$oid': '5160757496cc6207a37ff778'},Pomegranate Yogurt Bowl,For each bowl: \na big dollop of Greek yogurt\...,http://www.101cookbooks.com/archives/pomegrana...,http://www.101cookbooks.com/mt-static/images/f...,{'$date': 1365276020318},NaN,101cookbooks,Serves 1.,2013-01-20,PT5M,A simple breakfast bowl made with Greek yogurt...,NaN,NaN,NaN,NaN,NaN


We see there are nearly 175,000 recipes, and 17 columns.
Let's take a look at one row to see what we have:

In [20]:
recipes.iloc[0]

,0
_id,{'$oid': '5160756b96cc62079cc2db15'}
name,Drop Biscuits and Sausage Gravy
ingredients,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...
url,http://thepioneerwoman.com/cooking/2013/03/dro...
image,http://static.thepioneerwoman.com/cooking/file...
ts,{'$date': 1365276011104}
cookTime,PT30M
source,thepioneerwoman
recipeYield,12
datePublished,2013-03-11


There is a lot of information there, but much of it is in a very messy form, as is typical of data scraped from the web.
In particular, the ingredient list is in string format; we're going to have to carefully extract the information we're interested in.
Let's start by taking a closer look at the ingredients:

In [21]:
recipes.ingredients.str.len().describe()

,ingredients
count,173278.000000
mean,244.617926
std,146.705285
min,0.000000
25%,147.000000
50%,221.000000
75%,314.000000
max,9067.000000


The ingredient lists average 250 characters long, with a minimum of 0 and a maximum of nearly 10,000 characters!

Just out of curiosity, let's see which recipe has the longest ingredient list:

In [22]:
recipes.name[np.argmax(recipes.ingredients.str.len())]

'Carrot Pineapple Spice &amp; Brownie Layer Cake with Whipped Cream &amp; Cream Cheese Frosting and Marzipan Carrots'

We can do other aggregate explorations; for example, we can see how many of the recipes are for breakfast foods (using regular expression syntax to match both lowercase and capital letters):

In [23]:
recipes.description.str.contains('[Bb]reakfast').sum()

3524

Or how many of the recipes list cinnamon as an ingredient:

In [24]:
recipes.ingredients.str.contains('[Cc]innamon').sum()

np.int64(10526)

We could even look to see whether any recipes misspell the ingredient as "cinamon":

In [25]:
recipes.ingredients.str.contains('[Cc]inamon').sum()

np.int64(11)

This is the type of data exploration that is possible with Pandas string tools.
It is data munging like this that Python really excels at.

### A Simple Recipe Recommender

Let's go a bit further, and start working on a simple recipe recommendation system: given a list of ingredients, we want to find any recipes that use all those ingredients.
While conceptually straightforward, the task is complicated by the heterogeneity of the data: there is no easy operation, for example, to extract a clean list of ingredients from each row.
So, we will cheat a bit: we'll start with a list of common ingredients, and simply search to see whether they are in each recipe's ingredient list.
For simplicity, let's just stick with herbs and spices for the time being:

In [26]:
spice_list = ['salt', 'pepper', 'oregano', 'sage', 'parsley',
              'rosemary', 'tarragon', 'thyme', 'paprika', 'cumin']

We can then build a Boolean `DataFrame` consisting of `True` and `False` values, indicating whether each ingredient appears in the list:

In [27]:
import re
spice_df = pd.DataFrame({
    spice: recipes.ingredients.str.contains(spice, re.IGNORECASE)
    for spice in spice_list})
spice_df.head()

,salt,pepper,oregano,sage,parsley,rosemary,tarragon,thyme,paprika,cumin
0,False,False,False,True,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,True,True,False,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False


Now, as an example, let's say we'd like to find a recipe that uses parsley, paprika, and tarragon.
We can compute this very quickly using the `query` method of ``DataFrame``s, discussed further in [High-Performance Pandas: `eval()` and `query()`](03.12-Performance-Eval-and-Query.ipynb):

In [28]:
selection = spice_df.query('parsley & paprika & tarragon')
len(selection)

10

We find only 10 recipes with this combination. Let's use the index returned by this selection to discover the names of those recipes:

In [29]:
recipes.name[selection.index]

,name
2069,"All cremat with a Little Gem, dandelion and wa..."
74964,Lobster with Thermidor butter
93768,Burton's Southern Fried Chicken with White Gravy
113926,Mijo's Slow Cooker Shredded Beef
137686,Asparagus Soup with Poached Eggs
140530,Fried Oyster Po’boys
158475,Lamb shank tagine with herb tabbouleh
158486,Southern fried chicken in buttermilk
163175,Fried Chicken Sliders with Pickles + Slaw
165243,Bar Tartine Cauliflower Salad


Now that we have narrowed down our recipe selection from 175,000 to 10, we are in a position to make a more informed decision about what we'd like to cook for dinner.

### Going Further with Recipes

Hopefully this example has given you a bit of a flavor (heh) of the types of data cleaning operations that are efficiently enabled by Pandas string methods.
Of course, building a robust recipe recommendation system would require a *lot* more work!
Extracting full ingredient lists from each recipe would be an important piece of the task; unfortunately, the wide variety of formats used makes this a relatively time-consuming process.
This points to the truism that in data science, cleaning and munging of real-world data often comprises the majority of the work—and Pandas provides the tools that can help you do this efficiently.

## **LLM(Large Language Model)**

In [30]:
recipes.head()

,_id,name,ingredients,url,image,ts,cookTime,source,recipeYield,datePublished,prepTime,description,totalTime,creator,recipeCategory,dateModified,recipeInstructions
0,{'$oid': '5160756b96cc62079cc2db15'},Drop Biscuits and Sausage Gravy,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,http://thepioneerwoman.com/cooking/2013/03/dro...,http://static.thepioneerwoman.com/cooking/file...,{'$date': 1365276011104},PT30M,thepioneerwoman,12,2013-03-11,PT10M,"Late Saturday afternoon, after Marlboro Man ha...",NaN,NaN,NaN,NaN,NaN
1,{'$oid': '5160756d96cc62079cc2db16'},Hot Roast Beef Sandwiches,12 whole Dinner Rolls Or Small Sandwich Buns (...,http://thepioneerwoman.com/cooking/2013/03/hot...,http://static.thepioneerwoman.com/cooking/file...,{'$date': 1365276013902},PT20M,thepioneerwoman,12,2013-03-13,PT20M,"When I was growing up, I participated in my Ep...",NaN,NaN,NaN,NaN,NaN
2,{'$oid': '5160756f96cc6207a37ff777'},Morrocan Carrot and Chickpea Salad,Dressing:\n1 tablespoon cumin seeds\n1/3 cup /...,http://www.101cookbooks.com/archives/moroccan-...,http://www.101cookbooks.com/mt-static/images/f...,{'$date': 1365276015332},NaN,101cookbooks,NaN,2013-01-07,PT15M,A beauty of a carrot salad - tricked out with ...,NaN,NaN,NaN,NaN,NaN
3,{'$oid': '5160757096cc62079cc2db17'},Mixed Berry Shortcake,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,http://thepioneerwoman.com/cooking/2013/03/mix...,http://static.thepioneerwoman.com/cooking/file...,{'$date': 1365276016700},PT15M,thepioneerwoman,8,2013-03-18,PT15M,It's Monday! It's a brand new week! The birds ...,NaN,NaN,NaN,NaN,NaN
4,{'$oid': '5160757496cc6207a37ff778'},Pomegranate Yogurt Bowl,For each bowl: \na big dollop of Greek yogurt\...,http://www.101cookbooks.com/archives/pomegrana...,http://www.101cookbooks.com/mt-static/images/f...,{'$date': 1365276020318},NaN,101cookbooks,Serves 1.,2013-01-20,PT5M,A simple breakfast bowl made with Greek yogurt...,NaN,NaN,NaN,NaN,NaN


In [31]:
recipes.columns

Index(['_id', 'name', 'ingredients', 'url', 'image', 'ts', 'cookTime',
       'source', 'recipeYield', 'datePublished', 'prepTime', 'description',
       'totalTime', 'creator', 'recipeCategory', 'dateModified',
       'recipeInstructions'],
      dtype='object')

In [32]:
# 필요한 컬럼만 추출

df = recipes[['name', 'ingredients', 'description',]]
df.head()

,name,ingredients,description
0,Drop Biscuits and Sausage Gravy,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,"Late Saturday afternoon, after Marlboro Man ha..."
1,Hot Roast Beef Sandwiches,12 whole Dinner Rolls Or Small Sandwich Buns (...,"When I was growing up, I participated in my Ep..."
2,Morrocan Carrot and Chickpea Salad,Dressing:\n1 tablespoon cumin seeds\n1/3 cup /...,A beauty of a carrot salad - tricked out with ...
3,Mixed Berry Shortcake,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,It's Monday! It's a brand new week! The birds ...
4,Pomegranate Yogurt Bowl,For each bowl: \na big dollop of Greek yogurt\...,A simple breakfast bowl made with Greek yogurt...


In [33]:
for text in df['ingredients'][:2]:
    print(text)
    print('*'*100)

Biscuits
3 cups All-purpose Flour
2 Tablespoons Baking Powder
1/2 teaspoon Salt
1-1/2 stick (3/4 Cup) Cold Butter, Cut Into Pieces
1-1/4 cup Butermilk
 SAUSAGE GRAVY
1 pound Breakfast Sausage, Hot Or Mild
1/3 cup All-purpose Flour
4 cups Whole Milk
1/2 teaspoon Seasoned Salt
2 teaspoons Black Pepper, More To Taste
****************************************************************************************************
12 whole Dinner Rolls Or Small Sandwich Buns (I Used Whole Wheat)
1 pound Thinly Shaved Roast Beef Or Ham (or Both!)
1 pound Cheese (Provolone, Swiss, Mozzarella, Even Cheez Whiz!)
1/4 cup Mayonnaise
3 Tablespoons Grated Onion (or 1 Tbsp Dried Onion Flakes))
1 Tablespoon Poppy Seeds
1 Tablespoon Spicy Mustard
1 Tablespoon Horseradish Mayo Or Straight Prepared Horseradish
 Dash Of Worcestershire
 Optional Dressing Ingredients: Sriracha, Hot Sauce, Dried Onion Flakes Instead Of Fresh, Garlic Powder, Pepper, Etc.)
******************************************************************

In [34]:
for text in df['description'][:5]:
    print(len(text))
    print(text)
    print('*'*100)

128
Late Saturday afternoon, after Marlboro Man had returned home with the soccer-playing girls, and I had returned home with the...
****************************************************************************************************
128
When I was growing up, I participated in my Episcopal church's youth group, and I have lots of memories of weekly meetings wh...
****************************************************************************************************
150
A beauty of a carrot salad - tricked out with chickpeas, chunks of dried pluots, sliced almonds, and a toasted cumin dressing. Thank you Diane Morgan.
****************************************************************************************************
128
It's Monday! It's a brand new week! The birds are chirping! The coffee's brewing! Everything has such hope and promise!     A...
****************************************************************************************************
130
A simple breakfast bowl made with Gree

In [35]:
print(df['description'][:1].values)

['Late Saturday afternoon, after Marlboro Man had returned home with the soccer-playing girls, and I had returned home with the...']


## **Google gemnini API Key로 파이썬 코딩하여 앱을 만들기 시작하는 방법**

### [Google AI Studio](https://aistudio.google.com/prompts/new_chat)로 이동하여 Google 계정으로 로그인합니다.
### [API 키](https://aistudio.google.com/app/apikey)를 만듭니다.(API 키를 발급받아 Colab에 key에 저장)
### [Python용](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started.ipynb) 빠른 시작을 사용하거나 curl을 사용하여 REST API를 호출합니다.

In [36]:
%pip install -U -q 'google-genai>=1.0.0'

In [37]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('gemini-key')

In [38]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

### Choose a model

Select the model you want to use in this guide. You can either select one from the list or enter a model name manually. Keep in mind that some models, such as the 2.5 ones are thinking models and thus take slightly more time to respond. For more details, you can see [thinking notebook](./Get_started_thinking.ipynb) to learn how to switch the thinking off.

For a full overview of all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini).

In [39]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro"] {"allow-input":true, isTemplate: true}

## Send text prompts

Use the `generate_content` method to generate responses to your prompts. You can pass text directly to `generate_content` and use the `.text` property to get the text content of the response. Note that the `.text` field will work when there's only one part in the output.

In [40]:
from IPython.display import Markdown

response = client.models.generate_content(
    model=MODEL_ID,
    contents="What's the largest planet in our solar system?"
)

Markdown(response.text)

The largest planet in our solar system is **Jupiter**.

In [41]:
# Prompts
'''
response = client.models.generate_content( model=MODEL_ID, contents="What's the largest planet in our solar system?" ) Markdown(response.text) 이 코드에서 contents를 input()문장으로 입력받아 수행하여다오
'''

'\nresponse = client.models.generate_content( model=MODEL_ID, contents="What\'s the largest planet in our solar system?" ) Markdown(response.text) 이 코드에서 contents를 input()문장으로 입력받아 수행하여다오\n'

In [42]:
from IPython.display import Markdown

user_input = input("Enter your prompt: ")

response = client.models.generate_content(
    model=MODEL_ID,
    contents=user_input
)

Markdown(response.text)

Enter your prompt: What's the largest planet in our solar system?


The largest planet in our solar system is **Jupiter**.

## **gradio UI로 코드해줘 (claude 로 코딩하길 권장)**

In [43]:
# %pip install gradio

In [44]:
import gradio as gr
from google import genai
from google.genai import types

# MODEL_ID = "gemini-1.5-flash"

client = genai.Client(api_key=GEMINI_API_KEY)

def generate_response(user_input):
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=user_input
        )
        return response.text
    except Exception as e:
        return f"오류: {str(e)}"

# Gradio 인터페이스
with gr.Blocks() as demo:
    gr.Markdown("# Gemini AI Chat")

    user_input = gr.Textbox(
        label="질문 입력",
        placeholder="질문을 입력하세요...",
        lines=3
    )

    output = gr.Textbox(
        label="AI 응답",
        lines=10,
        interactive=False
    )

    submit_btn = gr.Button("응답 생성", variant="primary")

    submit_btn.click(
        fn=generate_response,
        inputs=[user_input],
        outputs=[output]
    )

    user_input.submit(
        fn=generate_response,
        inputs=[user_input],
        outputs=[output]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30f1ef0fc226c45c30.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [45]:
df.head()

,name,ingredients,description
0,Drop Biscuits and Sausage Gravy,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,"Late Saturday afternoon, after Marlboro Man ha..."
1,Hot Roast Beef Sandwiches,12 whole Dinner Rolls Or Small Sandwich Buns (...,"When I was growing up, I participated in my Ep..."
2,Morrocan Carrot and Chickpea Salad,Dressing:\n1 tablespoon cumin seeds\n1/3 cup /...,A beauty of a carrot salad - tricked out with ...
3,Mixed Berry Shortcake,Biscuits\n3 cups All-purpose Flour\n2 Tablespo...,It's Monday! It's a brand new week! The birds ...
4,Pomegranate Yogurt Bowl,For each bowl: \na big dollop of Greek yogurt\...,A simple breakfast bowl made with Greek yogurt...


In [46]:
df['description'][0]

'Late Saturday afternoon, after Marlboro Man had returned home with the soccer-playing girls, and I had returned home with the...'

In [47]:
## 이 문장을 넣고 요약해달라고 prompt 해보시길